# OpenWorld — 60-second Quickstart

[OpenWorld](https://github.com/quome-cloud/openworld) is the *PyTorch for code world models*: describe a world, get **verified Python dynamics**, and run it deterministically — **no training, no GPU, no dataset**.

This notebook defines a world, runs it, serializes it to a portable spec, and renders its **model card** — all offline, no LLM required.

**Run all cells** (Runtime → Run all). If it's useful, please [⭐ star the repo](https://github.com/quome-cloud/openworld).

In [ ]:
# The core is zero-dependency; install straight from GitHub (not yet on PyPI).
!pip install -q "git+https://github.com/quome-cloud/openworld.git"

In [ ]:
from openworld import World, Action, CodeTransition, to_spec, render_card

DYNAMICS = '''
def transition(state, action):
    s = dict(state)
    if action["name"] == "heat":   s["temp"] += 1
    elif action["name"] == "cool": s["temp"] -= 1
    return s            # 'idle' (and anything else) holds -- explicit & verifiable
'''

room = World(
    name="thermostat",
    description="A room with a thermostat tracking a target temperature.",
    initial_state={"temp": 18, "target": 21},
    actions=["heat", "cool", "idle"],
    rules=["'heat' raises temp by 1, 'cool' lowers it by 1, 'idle' holds."],
    transition=CodeTransition(DYNAMICS),       # verified code -- not a neural net
)

print(room.transition.step(room.initial_state, Action("heat")))  # {'temp': 19, 'target': 21}
print(to_spec(room)["state_schema"])                             # {'temp': 'int', 'target': 'int'}

In [ ]:
# Render the self-contained SVG model card and show it inline.
from IPython.display import SVG, display
render_card(room, "thermostat.svg")
display(SVG(filename="thermostat.svg"))

That card — state graph, schema, verified dynamics, rollout, metrics, and the declared rule contract — is generated from the one `World` object.

**Next:** compose worlds-within-worlds, steer objectives with dials, or `openworld serve specs/ --allow-code` to deploy any spec as a live inference server. See the [README](https://github.com/quome-cloud/openworld) and [`experiments/`](https://github.com/quome-cloud/openworld/tree/main/experiments).